# 🪶 Flyweight Design Pattern — Hinglish mein samjho (Beginner Friendly)

Is notebook mein hum **Flyweight Design Pattern** ko step-by-step, Hinglish comments ke saath samjhenge.
Har code cell ke andar ek **multiline comment (docstring)** hai jisme concept Hinglish mein explain kiya gaya hai.

Pehle dekhenge **bina pattern ke** kya problem aati hai, phir **Flyweight Pattern** se use solve karenge.
Aakhir mein ek **Practice Question** aur uska **Coding Solution** bhi diya gaya hai taaki tum khud practice kar sako.

> Note: Original documentation mein example Java mein tha (Icon system ke liye) — humne wahi concept Python mein implement kiya hai.

In [1]:
"""
FLYWEIGHT DESIGN PATTERN — KYA HAI?
========================================

Flyweight ek STRUCTURAL design pattern hai jo MEMORY USAGE KAM
karta hai, COMMON (SHARED) data ko MULTIPLE objects ke BEECH
SHARE karke -- HAR object apna KHUD KA COPY nahi rakhta.

Ye TAB especially USEFUL hai jab APPLICATION ko BAHUT SAARE
SIMILAR objects BANANE PADEIN.

SOCHO EK EXAMPLE:
--------------------
Ek Game mein HAZAARON "trees" hain. Agar HAR tree apna KHUD KA
mesh + texture data (jo MB mein ho sakta hai) STORE kare, to
MEMORY EXPLODE ho jaayegi -- jabki SAARE Oak Trees ka
MESH/TEXTURE SAME hai, sirf unki POSITION ALAG hai!

Flyweight isko DO PARTS mein BAANT deta hai:
    1. INTRINSIC STATE (SHARED)   -> Jo data SAARE SIMILAR
                                    objects mein SAME hai (mesh,
                                    texture) -- ISKO EK BAAR
                                    STORE karo aur SHARE karo
    2. EXTRINSIC STATE (UNIQUE)     -> Jo data HAR OBJECT ke
                                    liye ALAG hai (position,
                                    speed) -- ISKO BAHAR, CLIENT
                                    ke paas STORE karo

MAIN COMPONENTS:
--------------------
1. Flyweight Interface   -> COMMON interface jo EXTRINSIC state
                           ACCEPT karta hai (e.g. Icon -- draw(x, y))
2. Concrete Flyweight       -> ACTUAL SHARED object, INTRINSIC
                           state STORE karta hai (e.g. FileIcon)
3. Flyweight Factory           -> Flyweight objects ka POOL/CACHE
                           MAINTAIN karta hai, REUSE ENSURE karta
                           hai (e.g. IconFactory)
4. Client                        -> Flyweight objects ko FACTORY
                           SE maangta hai, EXTRINSIC state (jaise
                           position) SUPPLY karta hai

KAISE KAAM KARTA HAI:
--------------------------
1. Client, Factory se ek Flyweight object MAANGTA hai
2. Factory CHECK karta hai -- "kya MATCHING flyweight PEHLE SE
   hai?" Agar HAAN, to WAHI RETURN karo
3. Agar NAHI, to Factory NAYA Flyweight BANATA hai, CACHE mein
   STORE karta hai, RETURN karta hai
4. Client, METHOD CALL ke waqt EXTRINSIC state (jaise x, y)
   SUPPLY karta hai

REAL LIFE EXAMPLES:
-----------------------
- Text Editor Characters: HAZAARON characters SAME font/size/style
  SHARE karte hain, sirf POSITION/CONTENT ALAG hai
- Traffic Signals: DIFFERENT intersections SAME signal-TYPES
  (red/yellow/green) REUSE karte hain
- Trees in a Forest Game: Tree MODELS SHARE hote hain, sirf
  POSITION/HEIGHT ALAG

FAYDE (ADVANTAGES):
------------------------
- COMMON object data SHARE karke MEMORY CONSUMPTION KAM hoti hai
- BAHUT SAARE SIMILAR objects wale apps mein PERFORMANCE IMPROVE
  hoti hai
- OBJECT CREATION OVERHEAD MINIMIZE hota hai

NUKSAAN (DISADVANTAGES):
----------------------------
- INTRINSIC aur EXTRINSIC data ko SEPARATE karna COMPLEXITY
  BADHATA hai
- SHARED aur UNIQUE states ko CAREFULLY DESIGN karna padta hai
- Agar objects HIGHLY UNIQUE hain (SHARE karne LAYAK KUCH NAHI
  hai), to ye pattern USEFUL NAHI hai
"""
print("Concept samajh liya! Ab neeche code mein implement karke dekhte hain.")


Concept samajh liya! Ab neeche code mein implement karke dekhte hain.


## ❌ Pehle dekho: BINA Flyweight Pattern ke (Problem Code)

Chalo pehle ye dekhte hain ki Flyweight Pattern KE BINA log GUI Icons (File/Folder icons) kaise
banate hain, aur usme kya dikkat aati hai.

In [2]:
"""
BINA FLYWEIGHT KE — PROBLEM DEMO
--------------------------------------
Yahan HAR icon object apna KHUD KA "image data" (simulate kiya
hai ek BADI STRING se, real app mein ye ACTUAL image bytes hote)
STORE karta hai -- CHAHE 100 file icons SAME image use kar rahe
hon, HAR EK apna ALAG COPY rakhta hai!
"""

class FileIconBad:
    def __init__(self, icon_type, image_data):
        self.icon_type = icon_type
        self.image_data = image_data   # HAR object apna KHUD KA COPY rakh raha hai!
        self.x = None
        self.y = None

    def draw(self, x, y):
        self.x, self.y = x, y
        print(f"Drawing {self.icon_type} icon at position ({x}, {y})")


# Socho ye ek BADA image data hai (real app mein KB/MB ka hota)
DUMMY_IMAGE_DATA = "🖼️" * 1000   # simulate ek "heavy" image payload

icons = []
for i in range(5):
    # HAR baar NAYA object, NAYA image_data COPY -- CHAHE SAB SAME "document" icon hain!
    icon = FileIconBad("document", DUMMY_IMAGE_DATA)
    icon.draw(x=i * 50, y=0)
    icons.append(icon)

print(f"\nTotal FileIconBad objects banaye gaye: {len(icons)}")
print(f"Har object apna ALAG image_data copy rakh raha hai (memory mein {len(icons)} copies hain, jabki data SAME hai)")


Drawing document icon at position (0, 0)
Drawing document icon at position (50, 0)
Drawing document icon at position (100, 0)
Drawing document icon at position (150, 0)
Drawing document icon at position (200, 0)

Total FileIconBad objects banaye gaye: 5
Har object apna ALAG image_data copy rakh raha hai (memory mein 5 copies hain, jabki data SAME hai)


In [3]:
"""
ISME PROBLEM KYA THI?
-------------------------
1. DUPLICATE MEMORY USAGE: Upar 5 icons banaye, aur SAARE
   "document" TYPE ke hain -- SAME image data. Phir bhi HAR EK
   ne apna ALAG "image_data" COPY STORE kiya. Agar image_data
   1 MB ka ho, to 5 icons = 5 MB, jabki ACTUALLY sirf 1 MB DATA
   hi UNIQUE hai!

2. SCALE NAHI HOTA: Agar ek Screen par 10,000 icons dikhane hon
   (jaise ek BADI file explorer window mein), to 10,000 SEPARATE
   image_data copies MEMORY mein honge -- APPLICATION CRASH ho
   sakti hai ya BAHUT SLOW ho sakti hai.

3. NO CENTRALIZED REUSE: Har jagah jahan "FileIconBad('document',
   ...)" call hota hai, NAYA OBJECT + NAYA DATA bante rehte hain
   -- KOI CHECK NAHI hai ki "kya ye ICON PEHLE SE EXIST karta hai?"

4. INTRINSIC aur EXTRINSIC DATA MIXED hain: "image_data" (SHARED
   hona chahiye) aur "x, y" (UNIQUE hona chahiye) DONO SAME
   object ke ANDAR MIXED hain -- ISI wajah se SHARING POSSIBLE
   NAHI hai.

YE SAB PROBLEMS "Flyweight Design Pattern" SOLVE KARTA HAI!
Neeche BEST APPROACH dekhte hain -- SHARED data (image) ko EK
BAAR STORE karenge, aur SAARE icons ISE SHARE karenge.
"""
print("Chalo ab BEST APPROACH (Flyweight Pattern) dekhte hain...")


Chalo ab BEST APPROACH (Flyweight Pattern) dekhte hain...


## ✅ Ab dekho: BEST APPROACH — Flyweight Pattern se solve karte hain

### 1️⃣ Flyweight Interface — `Icon`

In [4]:
"""
FLYWEIGHT INTERFACE (Icon)
--------------------------------
Ye COMMON interface hai jise SAARE Flyweight objects follow
karenge. "draw(x, y)" method EXTRINSIC state (x, y position) ko
PARAMETER ki tarah ACCEPT karta hai -- object ke ANDAR STORE
NAHI hota.
"""

from abc import ABC, abstractmethod

class Icon(ABC):
    @abstractmethod
    def draw(self, x, y):
        pass


### 2️⃣ Concrete Flyweights — `FileIcon`, `FolderIcon`

In [5]:
"""
CONCRETE FLYWEIGHT CLASSES (FileIcon, FolderIcon)
--------------------------------------------------------
Ye classes SIRF INTRINSIC (SHARED) state STORE karti hain --
"type"/"color" aur "image_name". POSITION (x, y) INKE ANDAR
STORE NAHI hota -- wo "draw()" call ke waqt PARAMETER ki tarah
AATA hai (EXTRINSIC state).
"""

class FileIcon(Icon):
    def __init__(self, icon_type, image_name):
        self.icon_type = icon_type       # INTRINSIC (shared)
        self.image_name = image_name     # INTRINSIC (shared)

    def draw(self, x, y):
        # x, y -- EXTRINSIC state, PARAMETER se aaya, object mein STORE NAHI
        print(f"Drawing {self.icon_type} icon with image {self.image_name} at position ({x}, {y})")


class FolderIcon(Icon):
    def __init__(self, color, image_name):
        self.color = color               # INTRINSIC (shared)
        self.image_name = image_name     # INTRINSIC (shared)

    def draw(self, x, y):
        print(f"Drawing folder icon with color {self.color} and image {self.image_name} at position ({x}, {y})")


### 3️⃣ Flyweight Factory — `IconFactory`

In [6]:
"""
FLYWEIGHT FACTORY (IconFactory)
--------------------------------------
Ye class Flyweight objects ka CACHE (dictionary) MAINTAIN karti
hai. "get_icon(key)" pehle CHECK karta hai ki ICON PEHLE SE
CACHE mein hai -- agar HAAN to WAHI RETURN karo, WARNA NAYA
banao, CACHE mein STORE karo, PHIR return karo.
"""

class IconFactory:
    def __init__(self):
        self.icon_cache = {}

    def get_icon(self, key):
        if key in self.icon_cache:
            print(f"♻️  '{key}' icon PEHLE SE cache mein hai -- REUSE ho raha hai")
            return self.icon_cache[key]

        # CACHE mein NAHI mila -- NAYA banao
        print(f"⏳ '{key}' icon PEHLI BAAR ban raha hai (SIRF EK BAAR)...")
        if key == "file":
            icon = FileIcon("document", "document.png")
        elif key == "folder":
            icon = FolderIcon("blue", "folder.png")
        else:
            raise ValueError(f"Unsupported icon type: {key}")

        self.icon_cache[key] = icon
        return icon


### 4️⃣ Client — Factory se icons maangta hai, POSITION supply karta hai

In [7]:
"""
CLIENT CODE
---------------
Client SEEDHA "FileIcon()" ya "FolderIcon()" NAHI banata -- wo
FACTORY se icon MAANGTA hai. Alag alag POSITIONS (EXTRINSIC
state) "draw()" call ke waqt PASS hote hain.
"""

icon_factory = IconFactory()

# File icons DIFFERENT positions par draw kar rahe hain
file_icon1 = icon_factory.get_icon("file")
file_icon1.draw(100, 100)

file_icon2 = icon_factory.get_icon("file")   # SAME object REUSE hoga!
file_icon2.draw(150, 150)

# Folder icons
folder_icon1 = icon_factory.get_icon("folder")
folder_icon1.draw(200, 200)

folder_icon2 = icon_factory.get_icon("folder")
folder_icon2.draw(250, 250)

print(f"\nKya file_icon1 aur file_icon2 SAME object hain?", file_icon1 is file_icon2)
print(f"Total UNIQUE Flyweight objects banaye gaye: {len(icon_factory.icon_cache)} (chahe 4 baar draw() call hua)")

print("\n✅ Flyweight Pattern se 4 draw() calls ke liye SIRF 2 ACTUAL icon objects bane -- memory bahut SAVE hui!")


⏳ 'file' icon PEHLI BAAR ban raha hai (SIRF EK BAAR)...
Drawing document icon with image document.png at position (100, 100)
♻️  'file' icon PEHLE SE cache mein hai -- REUSE ho raha hai
Drawing document icon with image document.png at position (150, 150)
⏳ 'folder' icon PEHLI BAAR ban raha hai (SIRF EK BAAR)...
Drawing folder icon with color blue and image folder.png at position (200, 200)
♻️  'folder' icon PEHLE SE cache mein hai -- REUSE ho raha hai
Drawing folder icon with color blue and image folder.png at position (250, 250)

Kya file_icon1 aur file_icon2 SAME object hain? True
Total UNIQUE Flyweight objects banaye gaye: 2 (chahe 4 baar draw() call hua)

✅ Flyweight Pattern se 4 draw() calls ke liye SIRF 2 ACTUAL icon objects bane -- memory bahut SAVE hui!


## 📝 Practice Question (Tumhari baari!)

**Problem Statement:**

Ek **Text Editor Character Rendering System** banao using Flyweight Design Pattern (jaise document
ke "Real-Life Example" mein "Text Editor Characters" mention hua tha).

Ek document mein HAZAARON characters hain, lekin SAARE characters LIMITED SET ki `font` aur `size`
combinations SHARE karte hain (jaise "Arial-12", "Arial-Bold-14"). Har character ki SIRF `position`
(row, column) UNIQUE honi chahiye.

Tumhe ye banana hai (Icon/IconFactory example jaisi hi structure use karo):

1. `CharacterStyle` (Flyweight interface) — abstract method `render(char, row, col)`
2. `ConcreteCharacterStyle` (Concrete Flyweight) — `font` aur `size` (INTRINSIC state) store kare; `render(char, row, col)` print kare `"'{char}' [{font}, {size}pt] at (row={row}, col={col})"`
3. `CharacterStyleFactory` (Flyweight Factory) — `get_style(font, size)` method jo CACHE check kare (key = `(font, size)`), agar NAHI mile to NAYA `ConcreteCharacterStyle` banaye aur cache kare
4. Client code likho jo:
   - Ek `CharacterStyleFactory` banaye
   - "Hello" word ke 5 characters ko SAME style ("Arial", 12) se render kare, HAR character ki position ALAG ho (col = 0,1,2,3,4)
   - Verify kare ki SAARE 5 characters SAME `ConcreteCharacterStyle` object SHARE kar rahe hain (`is` operator se)

👉 Pehle khud try karo neeche wale khaali code cell mein, phir uske baad wale cell mein solution dekho!

In [8]:
"""
YAHAN KHUD LIKHO (Practice Cell)
-----------------------------------
Hint: Icon/FileIcon/IconFactory example ko copy karke sirf naam
badal do:
Icon -> CharacterStyle, FileIcon -> ConcreteCharacterStyle
IconFactory -> CharacterStyleFactory, draw(x,y) -> render(char, row, col)

Apna code neeche likho:
"""

# TODO: class CharacterStyle (ABC) banao with abstract render(char, row, col)
# TODO: class ConcreteCharacterStyle banao with font, size (INTRINSIC)
# TODO: class CharacterStyleFactory banao with get_style(font, size) aur cache
# TODO: Client code likho: "Hello" ke 5 characters render karo, SAME style ke saath
# TODO: Verify karo ki saare characters SAME style object share kar rahe hain


'\nYAHAN KHUD LIKHO (Practice Cell)\n-----------------------------------\nHint: Icon/FileIcon/IconFactory example ko copy karke sirf naam\nbadal do:\nIcon -> CharacterStyle, FileIcon -> ConcreteCharacterStyle\nIconFactory -> CharacterStyleFactory, draw(x,y) -> render(char, row, col)\n\nApna code neeche likho:\n'

## ✅ Solution — Practice Question ka Coding Solution

In [9]:
"""
SOLUTION EXPLANATION
-----------------------
Yahan bilkul Icon/FileIcon/IconFactory jaisi hi structure use ki
hai, bas ab "Character Styles" bana rahe hain instead of "Icons".

1. CharacterStyle             -> Flyweight Interface
2. ConcreteCharacterStyle        -> Concrete Flyweight (INTRINSIC: font, size)
3. CharacterStyleFactory            -> Flyweight Factory (cache font+size combos)
4. Neeche client code hai jo "Hello" render karta hai
"""

from abc import ABC, abstractmethod

# ---- 1. Flyweight Interface ----
class CharacterStyle(ABC):
    @abstractmethod
    def render(self, char, row, col):
        pass


# ---- 2. Concrete Flyweight ----
class ConcreteCharacterStyle(CharacterStyle):
    def __init__(self, font, size):
        self.font = font     # INTRINSIC (shared)
        self.size = size     # INTRINSIC (shared)

    def render(self, char, row, col):
        # char, row, col -- EXTRINSIC state, object mein STORE NAHI hota
        print(f"'{char}' [{self.font}, {self.size}pt] at (row={row}, col={col})")


# ---- 3. Flyweight Factory ----
class CharacterStyleFactory:
    def __init__(self):
        self.style_cache = {}

    def get_style(self, font, size):
        key = (font, size)
        if key not in self.style_cache:
            print(f"⏳ Naya style '{font}-{size}pt' ban raha hai (SIRF EK BAAR)...")
            self.style_cache[key] = ConcreteCharacterStyle(font, size)
        else:
            print(f"♻️  Style '{font}-{size}pt' REUSE ho raha hai")
        return self.style_cache[key]


# ---- 4. Client Code ----
style_factory = CharacterStyleFactory()
word = "Hello"

rendered_styles = []
for col, char in enumerate(word):
    style = style_factory.get_style("Arial", 12)
    style.render(char, row=0, col=col)
    rendered_styles.append(style)

print(f"\nTotal ConcreteCharacterStyle objects banaye gaye: {len(style_factory.style_cache)}")
print("Kya saare 5 characters SAME style object share kar rahe hain?", all(s is rendered_styles[0] for s in rendered_styles))


⏳ Naya style 'Arial-12pt' ban raha hai (SIRF EK BAAR)...
'H' [Arial, 12pt] at (row=0, col=0)
♻️  Style 'Arial-12pt' REUSE ho raha hai
'e' [Arial, 12pt] at (row=0, col=1)
♻️  Style 'Arial-12pt' REUSE ho raha hai
'l' [Arial, 12pt] at (row=0, col=2)
♻️  Style 'Arial-12pt' REUSE ho raha hai
'l' [Arial, 12pt] at (row=0, col=3)
♻️  Style 'Arial-12pt' REUSE ho raha hai
'o' [Arial, 12pt] at (row=0, col=4)

Total ConcreteCharacterStyle objects banaye gaye: 1
Kya saare 5 characters SAME style object share kar rahe hain? True


## 🎯 Bonus Challenge

Ab try karo:
1. Ek AUR word "World" ko `"Arial-Bold"`, size 14 se render karo — verify karo ki ab
   `CharacterStyleFactory` mein TOTAL 2 unique styles hain (Arial-12 aur Arial-Bold-14), chahe
   10 characters render hue hon.
2. Socho: agar 1 lakh characters wala EK BADA document ho, jisme SIRF 5 DIFFERENT (font, size)
   combinations use ho rahe hon, to Flyweight Pattern KITNI MEMORY SAVE karega (BINA Flyweight
   ke 1,00,000 style objects vs Flyweight ke SATH sirf 5)?
3. `ConcreteCharacterStyle` mein ek naya INTRINSIC attribute `color` add karo — dekho `get_style()`
   ka `key` tuple mein bhi `color` add karna padega, taaki ALAG COLOR wale characters ALAG
   style share NA karein.